In [ ]:
import duckdb

In [ ]:
query_hvfhv_2019_2023 = """
WITH CTE_hvfhv_2019_2023 AS (
    SELECT 
        hvfhs_license_num AS provider,
        request_datetime AS request_time,
        pickup_datetime AS pick_up_time,
        CAST(trip_miles AS FLOAT) AS trip_distance,
        CAST(trip_time AS INTEGER) AS duration_seconds,
        CAST(base_passenger_fare AS FLOAT) AS base_fare,
        CAST(tolls AS FLOAT) AS toll_fare,
        CAST(bcf AS FLOAT) AS bcf_fare,
        CAST(sales_tax AS FLOAT) AS tax_fare,
        CAST(tips AS FLOAT) AS tip_amount,
        shared_request_flag AS shared_before,
        shared_match_flag AS shared_during,
        wav_request_flag AS wheelchair_request,
        CAST( CASE tips
            WHEN  0.0 THEN 1
            ELSE 0
        END AS INTEGER) AS tip_category
    FROM 'C:/Users/ekadw/Documents/DATA/NY_Taxi/*/high_volume_for_hire_vehicle/fhvhv_tripdata_*.parquet'
    WHERE hvfhs_license_num IS NOT NULL
        AND request_datetime IS NOT NULL
        AND pickup_datetime IS NOT NULL
        AND trip_miles >= 0
        AND trip_miles <= 50
        AND trip_time >= 0
        AND base_passenger_fare >= 0
        AND tolls >= 0
        AND bcf >= 0
        AND sales_tax >= 0
        AND tips >= 0
        AND shared_request_flag IS NOT NULL
        AND shared_match_flag IS NOT NULL
        AND wav_request_flag IS NOT NULL
        AND request_datetime >= '2019-02-01' 
        AND request_datetime < '2023-10-01'
), CTE_duration_hvfhv_2019_2023 AS (
    SELECT
        provider,
        DATE_DIFF('day', request_time, pick_up_time) AS duration_days,
        EPOCH(pick_up_time - request_time) AS duration_request,
        trip_distance,
        duration_seconds,
        base_fare + toll_fare + bcf_fare + tax_fare + tip_amount AS total_amount,
        shared_before,
        shared_during,
        wheelchair_request,
        tip_category
    FROM CTE_hvfhv_2019_2023
)

SELECT 
    provider,
    duration_request,
    trip_distance,
    duration_seconds,
    total_amount,
    shared_before,
    shared_during,
    wheelchair_request,
    tip_category
FROM CTE_duration_hvfhv_2019_2023
WHERE duration_days = 0
"""

In [ ]:
import pandas as pd
from river import preprocessing, tree, metrics

# ----------------------------
# 1. Connect to DuckDB and query
# ----------------------------
con = duckdb.connect("my_data.duckdb")
res = con.execute(query_yellow_green)

# ----------------------------
# 2. Build pipeline
# ----------------------------
pipeline = preprocessing.StandardScaler() | tree.HoeffdingTreeClassifier()

# Compact classification report table
all_metrics = metrics.ClassificationReport()

downsample_ratio = 3  # keep 1:3 balance

# ----------------------------
# 2. Build pipeline
# ----------------------------
pipeline = (
    preprocessing.OneHotEncoder() |        # handle categorical features
    preprocessing.StandardScaler() |       # scale numerical features
    tree.HoeffdingTreeClassifier()         # online decision tree
)

all_metrics = metrics.ClassificationReport()
downsample_ratio = 3  # keep 1:3 balance

# ----------------------------
# 3. Streaming loop
# ----------------------------
while True:
    chunk = res.fetch_df_chunk(vectors_per_chunk=100_000)
    if chunk is None or len(chunk) == 0:
        break

    # Balance the batch
    minority = chunk[chunk["tip_category"] == 1]
    majority = chunk[chunk["tip_category"] == 0]

    if len(minority) > 0:
        majority_down = majority.sample(
            n=min(len(majority), downsample_ratio * len(minority)),
            random_state=42
        )
        batch_balanced = pd.concat([minority, majority_down], ignore_index=True)
    else:
        batch_balanced = chunk

    # Convert to dicts for speed
    records = batch_balanced.to_dict(orient="records")

    for r in records:
        x = {k: v for k, v in r.items() if k != "tip_category"}
        y = r["tip_category"]

        # Predict
        y_pred = pipeline.predict_one(x)

        # Update metrics
        if y_pred is not None:
            all_metrics.update(y, y_pred)

        # Train
        pipeline.learn_one(x, y)

    # Print compact classification report after each chunk
    print(all_metrics)